# Kaam-Vaani LoRA fine-tune — profile extraction

Fine-tunes **Qwen2.5-0.5B-Instruct** with LoRA on the 55 gold profile-extraction examples built by `scripts/build_lora_data.py` (49 train / 6 val). The goal is narrow and measurable: fix the semantic field errors the base 0.5B model makes on unseen transcripts (Step 8.7 / L60 — it put the goal value in `interests` and answered `goal_type` "explore" for a person who explicitly wants a job).

**This notebook was NOT executed.** It is a prepared path (Step 8.8): the training data builder ran and is tested, but no GPU exists in the sandbox where it was written. Run it on a **free Colab T4** (Runtime > Change runtime type > T4 GPU) or a free Kaggle GPU notebook. See `docs/LORA_RUNBOOK.md` before running.

Rules for the run:
- Nothing is uploaded to Hugging Face or anywhere else. The adapter is saved to the Colab runtime (download it, or copy it to Drive yourself).
- The last cell EVALUATES the adapter against the base model on the same val set. Do not claim an improvement without those numbers.
- If the run diverges or underfits, change the learning rate / epochs in `lora_config.yaml` first and record what worked in the runbook.

In [ ]:
# Install the training stack (TRL + PEFT + Transformers).
!pip install -q trl peft transformers datasets accelerate pyyaml

In [ ]:
# Fail fast without a GPU: CPU training of even a 0.5B model is not worth it.
import torch

assert torch.cuda.is_available(), "No GPU — switch the runtime to a T4 GPU first."
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

In [ ]:
# Get the training data: upload kv_lora_train.jsonl, kv_lora_val.jsonl
# and lora_config.yaml (from the repo's training/ directory) when prompted.
# Alternative: clone the repo instead of uploading —
#   !git clone https://github.com/Thunderdoc/career_guidance_llm_app.git
#   %cd career_guidance_llm_app && git checkout kaam-vaani
from google.colab import files

uploaded = files.upload()
for name in ("kv_lora_train.jsonl", "kv_lora_val.jsonl", "lora_config.yaml"):
    assert name in uploaded, f"missing upload: {name}"
print("uploads OK:", sorted(uploaded))

In [ ]:
# Load the config + the JSONL datasets.
import yaml
from datasets import load_dataset

with open("lora_config.yaml") as f:
    cfg = yaml.safe_load(f)

train_ds = load_dataset("json", data_files="kv_lora_train.jsonl", split="train")
val_ds = load_dataset("json", data_files="kv_lora_val.jsonl", split="train")
print(f"train: {len(train_ds)}  val: {len(val_ds)}  base: {cfg['base_model']}")

In [ ]:
# Convert messages -> prompt/completion pairs. The prompt is the
# system + user turns rendered with the chat template; the
# completion is the assistant's profile JSON. TRL trains on the
# completion only (completion_only_loss), so the model learns to
# PRODUCE the JSON, not to repeat the constant system prompt.
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(cfg["base_model"])
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


def to_prompt_completion(example):
    messages = example["messages"]
    prompt = tokenizer.apply_chat_template(messages[:2], tokenize=False, add_generation_prompt=True)
    completion = messages[2]["content"] + tokenizer.eos_token
    return {"prompt": prompt, "completion": completion}


train_pc = train_ds.map(to_prompt_completion, remove_columns=train_ds.column_names)
print(train_pc[0]["prompt"][-300:])
print("COMPLETION:", train_pc[0]["completion"][:200])

In [ ]:
# Load the base model + apply the LoRA config.
import torch
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    cfg["base_model"], torch_dtype=torch.bfloat16, attn_implementation="sdpa"
)
model.config.use_cache = False  # required with gradient checkpointing
lora = cfg["lora"]
peft_config = LoraConfig(
    r=lora["r"],
    lora_alpha=lora["alpha"],
    lora_dropout=lora["dropout"],
    target_modules=lora["target_modules"],
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

In [ ]:
# Train. Small data (49 examples), small model (0.5B): this is a
# short run on a T4. Watch the loss — it should fall steadily;
# if it spikes or flatlines, stop and adjust the learning rate.
from trl import SFTConfig, SFTTrainer

t = cfg["training"]
sft_config = SFTConfig(
    output_dir=cfg["output_dir"],
    num_train_epochs=t["epochs"],
    per_device_train_batch_size=t["per_device_train_batch_size"],
    gradient_accumulation_steps=t["gradient_accumulation_steps"],
    learning_rate=t["learning_rate"],
    max_length=t["max_length"],
    logging_steps=t["logging_steps"],
    save_strategy=t["save_strategy"],
    bf16=t["bf16"],
    gradient_checkpointing=t["gradient_checkpointing"],
    completion_only_loss=t["completion_only_loss"],
    report_to=[],  # no external logging; the loss prints here
)
trainer = SFTTrainer(model=model, args=sft_config, train_dataset=train_pc)
trainer.train()

In [ ]:
# Save the adapter (LoRA weights only, a few MB — not the full model).
# Download it from the file browser, or copy it to Drive yourself.
# Do NOT push it to the Hugging Face Hub without the owner's say-so.
trainer.save_model(cfg["output_dir"] + "/final")
tokenizer.save_pretrained(cfg["output_dir"] + "/final")
print("adapter saved to", cfg["output_dir"] + "/final")

In [ ]:
# EVALUATE: adapter vs base model on the same val set, greedy decoding.
# Per-field accuracy (skills as a set, like eval/run_kv.py) + exact-match
# rate. The adapter only counts as an improvement if it beats the base
# on this val set — record both rows in EVAL_REPORT.md either way.
import json
from contextlib import nullcontext as _nullcontext

import torch

FIELDS = ("skills", "interests", "education", "city", "income_text", "income_monthly", "goal_type")


def generate(prompt):
    device = next(model.parameters()).device
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=256, do_sample=False)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True)


def evaluate(use_adapter):
    field_hits = {f: 0 for f in FIELDS}
    exact, parsed = 0, 0
    ctx = model.disable_adapter() if not use_adapter else _nullcontext()
    with ctx:
        for ex in val_ds:
            prompt = tokenizer.apply_chat_template(
                ex["messages"][:2], tokenize=False, add_generation_prompt=True
            )
            target = json.loads(ex["messages"][2]["content"])
            try:
                got = json.loads(generate(prompt))
                parsed += 1
            except json.JSONDecodeError:
                continue
            ok = True
            for f in FIELDS:
                if f == "skills":
                    match = set(got.get(f, [])) == set(target[f])
                else:
                    match = got.get(f) == target[f]
                field_hits[f] += int(match)
                ok = ok and match
            exact += int(ok)
    n = len(val_ds)
    return {"parsed": parsed, "exact": exact, **{f"{f}": field_hits[f] for f in FIELDS}, "n": n}


base_scores = evaluate(use_adapter=False)
adapter_scores = evaluate(use_adapter=True)
print(f"{'field':<16}{'base':>6}{'adapter':>9}   (n={len(val_ds)})")
for key in ("parsed", "exact", *FIELDS):
    print(f"{key:<16}{base_scores[key]:>6}{adapter_scores[key]:>9}")

## After the run

1. **Record the numbers** — both rows (base + adapter) go into `EVAL_REPORT.md` with the date, the config used, and the val size (n=6). A 6-example val set is small: treat a win as a signal, not proof.
2. **The adapter is a candidate, not a ship.** Loading it in the app means merging or serving it with llama.cpp (`llama.cpp` can apply a LoRA adapter at load, or the adapter can be merged into the base model with PEFT's `merge_and_unload` and re-exported to GGUF). That step is NOT in this notebook — do it only after the val numbers justify it, and re-run the 8.7 live extraction check with the merged model.
3. **If the adapter does not beat the base**, the honest result is "LoRA on 55 examples did not help" — grow the dataset (see `docs/LORA_RUNBOOK.md`) before retrying with more epochs.